# Dapr

Dieses Tutorial hilft Ihnen beim Einrichten von Dapr in einem Kubernetes-Cluster. 

Zusammenfassend generiert die Python-Anwendung Nachrichten, die von der Node-Anwendung verarbeitet und gespeichert werden. Das folgende Architekturdiagramm veranschaulicht die Komponenten dieses Schnellstarts:

![](https://github.com/dapr/quickstarts/raw/master/tutorials/hello-kubernetes/img/Architecture_Diagram.png)

Quelle: [dapr.io](https://docs.dapr.io/concepts/overview/)

- - -

Installation mit helm (für Produktion)

    helm repo add dapr https://dapr.github.io/helm-charts/
    helm upgrade --install dapr dapr/dapr --version=1.16 --namespace dapr-system --create-namespace --wait
    helm install dapr-dashboard dapr/dapr-dashboard --namespace dapr-system

Installation mit dem Dapr CLI (für Entwicklung/Tests)  
* [Quelle](https://github.com/dapr/quickstarts/tree/master/tutorials/hello-kubernetes)

**ACHTUNG**: die Installationen dürfen nicht kombiniert werden, ansonsten treten Seiteneffekte durch Fehlkonfigurationen auf.


In [ ]:
%%bash
wget -q https://raw.githubusercontent.com/dapr/cli/master/install/install.sh -O - | /bin/bash
dapr init -k --dev

### UIs

* K8s Dashboard
* Dapr Dashboard
* Zipin - Kommunikationswege anzeigen

In [ ]:
%%bash
kubectl patch svc dapr-dashboard -n dapr-system -p '{"spec": {"type": "NodePort"}}'
kubectl patch svc dapr-dev-zipkin -n default -p '{"spec": {"type": "NodePort"}}'

echo "---------- Dashboards ---------"

echo "K8s Dashboard   : https://$(cat ~/data/server-ip):30443"
echo "Dapr Dashboard  : http://"$(cat ~/data/server-ip)":$(kubectl get -n dapr-system service dapr-dashboard -o=jsonpath='{ .spec.ports[0].nodePort }')/"
echo "Zipkin          : http://$(cat ~/data/server-ip):$(kubectl get -n default service dapr-dev-zipkin -o jsonpath='{.spec.ports[0].nodePort}')/"

echo "---------- Dapr Komponenten --------"
dapr status -k
kubectl get pods 


---
## Die/der Node.js Anwendung/Microservice

Zuerst holen wir uns das Repository.

In [ ]:
%%bash
git clone https://github.com/dapr/quickstarts.git

Deployment node.js Anwendung 

In [ ]:
%%bash
cd quickstarts/tutorials/hello-kubernetes
kubectl apply -f ./deploy/node.yaml
kubectl get pods,services -l app=node

Funktionstest: Node.js Microservice -> Dapr

In [ ]:
%%bash
curl "http://$(cat ~/data/server-ip):$(kubectl get service nodeapp -o jsonpath='{.spec.ports[0].nodePort}')/"ports

Als Nächstes können Sie eine Bestellung über die App aufgeben.

In [ ]:
%%bash
curl -X POST -vvv \
  -d '{"data":{"orderId":"43"}}' \
  -H "Content-Type: application/json" \
  http://$(cat ~/data/server-ip):$(kubectl get service nodeapp -o jsonpath='{.spec.ports[0].nodePort}')/neworder

Kontrolle durch Anzeigen der Bestellungen

In [ ]:
%%bash
curl http://$(cat ~/data/server-ip):$(kubectl get service nodeapp -o jsonpath='{.spec.ports[0].nodePort}')/order | jq

### Redis DB (unser State Store!) anschauen

In [ ]:
%%bash
kubectl exec $(kubectl get pods -l app.kubernetes.io/name=redis -o name) -- redis-cli -a $(kubectl get secret dapr-dev-redis   -o jsonpath="{.data.redis-password}" | base64 -d) scan 0

---
### Die Python-Anwendung mit dem Dapr-Sidecar

Der Code sieht so aus:

        dapr_http_endpoint = os.getenv("DAPR_HTTP_ENDPOINT", "http://localhost:3500")
        dapr_url = "{}/neworder".format(dapr_http_endpoint)
        
        n = 0
        while True:
            n += 1
            message = {"data": {"orderId": n}}
        
            try:
                response = requests.post(dapr_url, json=message)
            except Exception as e:
                print(e)
        
            time.sleep(1)

Was Dapr macht:
* Python-App sendet HTTP → localhost:3500
* Dapr Sidecar empfängt Request
* Sidecar erkennt Header dapr-app-id: nodeapp
* Dapr fragt Kubernetes (Name Resolution)
* Dapr findet Pod(s) von nodeapp
* Request wird an nodeapp weitergeleitet
* Response kommt zurück über den Sidecar
            

In [ ]:
%%bash
cd quickstarts/tutorials/hello-kubernetes
kubectl apply -f ./deploy/python.yaml

Die Bestellungen sollten nun im Logs ersichtlich sein

In [ ]:
%%bash
kubectl logs --selector=app=node -c node --tail=10

---
## Aufräumen

In [ ]:
%%bash
cd quickstarts/tutorials/hello-kubernetes
kubectl delete -f ./deploy/python.yaml || true
kubectl delete -f ./deploy/node.yaml || true
